MachineLearning을 위한 기본 컨셉을 예제를 통해 실습해본다.(2026/03/11 실습)
- 단변량->다변량 학습
-경사하강법, linearregression


In [39]:
# Graident-Descent-Impementing with Numpy

import numpy as np   
import matplotlib.pyplot as plt


#Data generation
np.random.seed(42)
x= np.random.rand(100,1)
y=1+2*x+1*np.random.randn(100,1)  #렌덤 노이즈를 추가한 이유 : 현실데이터는 완벽하지 않아서..

#Shuffles the indices
idx = np.arange(100)
np.random.shuffle(idx)

# Uses first 80 random indices for train ==> 훈련하기 위해서
train_idx = idx[:80]

# Uses the remaining indices for validation ==>  나머지는 검증용 validation데이타
val_idx = idx[80:]

# train을 위하 x, y에 값넣기 - 실제 데이터 넣기
x_train, y_train = x[train_idx], y[train_idx]
x_val, y_val = x[val_idx], y[val_idx]

#Data generation
np.random.seed(42)
a = np.random.randn(1) #세타0
b = np.random.randn(1) #세타1

print("세타0, 세타1:", a,b)
# 세타0, 세타1: [0.49671415] [-0.1382643]
#Set learning rate
lr = 1e-1    # 너무 큰 값이면 들쭉 날쭉한 결과로 발산 할수가 있다.(수렴에 가까워지게 만드는게 목표)

#Defines number of epochs
n_epochs = 1000

for epoch in range(n_epochs) :
    # Computes our model's predicted output
    yhat = a + b * x_train   # 예측치 : 세타0 + 세카1*x

    #How wrong is our model? That's the error!
    error = (y_train - yhat)    # 오차 = 결과 - 예측치

    # It is a regression, so it compute mean squared error(MSE)
    loss = (error**2).mean()   # 오차가 음의값도 나올수가 있어서 2배를 한 후 평균값을 구한다.
    
    # Computes gradients for both "a" and "b" parameters
    a_grad = -2 * error.mean()                      # a의 gradient 값
    b_grad = -2 * (x_train * error).mean()          # b의 gradient 값

    # Updates parameters using gradients and the learning rate
    a = a - lr * a_grad
    b = b - lr * b_grad

print(a,b)
#[1.23540769] [1.68964442]

#Sanity check : do we get the same results as our gradient desecnt?
from sklearn.linear_model import LinearRegression
linr = LinearRegression()
linr.fit(x_train, y_train)
print("sklearn.linear_model : ", linr.intercept_, linr.coef_[0])  #[1.23540755] [1.6896447]
#sklearn.linear_model :  [1.23540755] [1.6896447]

# 우리가 직접 만든 선형회귀와 실제 라이브러리 결과 비교.
# 위의 두 예제를 통해 우리가 만든 AI와 실제 머신러닝 라이브러리가 같음을 알자~


#======================================================================

#Pytorch:Autograd
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# numpy 데이터를 PyTorch 텐서로 변환
x_train_tensor = torch.from_numpy(x_train).float().to(device)
y_train_tensor = torch.from_numpy(y_train).float().to(device)

# a, b를 requires_grad=True인 PyTorch 텐서로 초기화
torch.manual_seed(42)
a = torch.randn(1, requires_grad=True, dtype=torch.float, device=device)
b = torch.randn(1, requires_grad=True, dtype=torch.float, device=device)

lr = 1e-1
n_epochs = 1000
for epoch in range(n_epochs):
    yhat = a + b * x_train_tensor
    error = y_train_tensor - yhat
    loss = (error**2).mean()

    # 역전파로 그래디언트 자동 계산
    loss.backward()

    with torch.no_grad():
        a -= lr * a.grad
        b -= lr * b.grad

    # 그래디언트 초기화 (zero_()는 in-place 연산)
    a.grad.zero_()
    b.grad.zero_()

print("PyTorch 결과:", a, b)
#PyTorch 결과: tensor([1.2354], requires_grad=True) tensor([1.6896], requires_grad=True)


#Optimizer : 파라메터 개수가 많을 때 Pythorch의 SGD, Adam과 같은 Optimizer를 써서 쉽게 해결함
#Gradient를 0으로 만드는 것도 Optimizer의 Zero_grad() method가 해결해줌
import torch.optim as optim

optimizer = optim.SGD([a,b], lr = lr)

for epoch in range(n_epochs):
    yhat = a + b * x_train_tensor
    error = y_train_tensor - yhat
    loss = (error**2).mean()

    # 역전파로 그래디언트 자동 계산
    loss.backward()

    optimizer.step()
    optimizer.zero_grad()

print("Optimizer 결과 : ", a, b)
#Optimizer 결과 :  tensor([1.2354], requires_grad=True) tensor([1.6896], requires_grad=True)

# Loss 계산 : mean squared error(MSE) loss가 적절하다
#nn.MSELoss는 손실함수를 만들어주는 것임
# 위의 optimizer가 있는 것하고 비교해 볼 필요가 있음
import torch.nn as nn

loss_fn = nn.MSELoss(reduction="mean")

optimizer = optim.SGD([a,b], lr = lr)

for epoch in range(n_epochs) :
    yhat = a + b * x_train_tensor

    loss = loss_fn(y_train_tensor, yhat)

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

print("Loss계산 결과 : ", a, b)
# Loss계산 결과 :  tensor([1.2354], requires_grad=True) tensor([1.6896], requires_grad=True)

## 다변량 선형회귀
class ManualLinearRegressor(nn.Module):
    def __init__(self):
        super().__init__()

        self.a = nn.Parameter(torch.rand(1, requires_grad=True, dtype = torch.float))
        self.b = nn.Parameter(torch.rand(1, requires_grad=True, dtype = torch.float))
    
    def forward(self, x):  
        return self.a + self.b*x

model = ManualLinearRegressor().to(device)

loss_fn = nn.MSELoss(reduction = 'mean')            #optimizer
optimizer = optim.SGD(model.parameters(), lr = lr)  #optimizer

for epoch in range(n_epochs):
    model.train()

    yhat = model(x_train_tensor)

    loss = loss_fn(yhat, y_train_tensor)
    loss.backward()                                 #update
    optimizer.step()
    optimizer.zero_grad()

print("ManualLinearRegressor for문 이후 model:", model.state_dict())
#다변량 선형회귀 for문 이후 model: OrderedDict({'a': tensor([1.2354]), 'b': tensor([1.6896])})

#수동으로 linear regression parameter를 생성하는 대신 pytorch의 Linear model을 만들어서 nested model을 생성
class LayerLinearRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(in_features=1, out_features=1)
    
    def forward(self, x):
        return self.linear(x)

#Training Step : loop를 조금 더 일반화 할 수 있는 방법
def make_train_step(model, loss_fn, optimizer):
    def train_step(x,y):
        # Sets model to train mode
        model.train()
        #Make predictions
        yhat = model(x)
        #Compute loss
        loss = loss_fn(yhat, y)
        #Compute gradients
        loss.backward()
        #Update parameters and zero gradients
        optimizer.step()
        optimizer.zero_grad()
        #Return the loss
        return loss.item()
    return train_step

#Sequential Models
model = nn.Sequential(nn.Linear(in_features = 1, out_features = 1)).to(device)
loss_fn = nn.MSELoss(reduction='mean')
optimizer = optim.SGD(model.parameters(), lr = lr)

train_step = make_train_step(model, loss_fn, optimizer)
losses = []

for epoch in range(n_epochs):
    loss = train_step(x_train_tensor, y_train_tensor)
    losses.append(loss)

print("make_train_step:LayerLinearRegressor for문 이후 :", model.state_dict())
#make_train_step:LayerLinearRegressor for문 이후 : OrderedDict({'0.weight': tensor([[1.6896]]), '0.bias': tensor([1.2354])})

#=======================================================================
# Gradient Descent in Practice
from torch.utils.data import Dataset, DataLoader, TensorDataset  # Dataset 추가
from torch.utils.data.dataset import random_split 
# Dataset 추가

class CustomDataset(Dataset):  
    def __init__(self, x_tensor, y_tensor):
        self.x = x_tensor
        self.y = y_tensor

    def __getitem__(self, index):
        return (self.x[index], self.y[index])
    
    def __len__(self):
        return len(self.x)  # x의 길이까지 모두 받아와



train_data = TensorDataset(x_train_tensor, y_train_tensor)
print(train_data[0])
#(tensor([0.7713]), tensor([1.8625]))
x_train_tensor = torch.from_numpy(x_train).float()
y_train_tensor = torch.from_numpy(y_train).float()

train_data = CustomDataset(x_train_tensor, y_train_tensor)  # class instance를 만들어
train_loader = DataLoader(dataset=train_data, batch_size=16, shuffle=True)
print("train_loader - ", train_data[0])  # train_date → train_data (오타 수정)
#train_loader -  (tensor([0.7713]), tensor([1.8625]))

next(iter(train_loader))

for epoch in range(n_epochs):
    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        loss = train_step(x_batch, y_batch)
        losses.append(loss)

x_tensor = torch.from_numpy(x).float()
y_tensor = torch.from_numpy(y).float()

dataset = TensorDataset(x_tensor, y_tensor)

train_dataset, valdataset = random_split(dataset, [80, 20])

train_loader = DataLoader(dataset=train_dataset, batch_size=16)
val_loader =  DataLoader(dataset=train_dataset, batch_size=20)

세타0, 세타1: [0.49671415] [-0.1382643]
[1.23540769] [1.68964442]
sklearn.linear_model :  [1.23540755] [1.6896447]
PyTorch 결과: tensor([1.2354], requires_grad=True) tensor([1.6896], requires_grad=True)
Optimizer 결과 :  tensor([1.2354], requires_grad=True) tensor([1.6896], requires_grad=True)
Loss계산 결과 :  tensor([1.2354], requires_grad=True) tensor([1.6896], requires_grad=True)
ManualLinearRegressor for문 이후 model: OrderedDict({'a': tensor([1.2354]), 'b': tensor([1.6896])})
make_train_step:LayerLinearRegressor for문 이후 : OrderedDict({'0.weight': tensor([[1.6896]]), '0.bias': tensor([1.2354])})
(tensor([0.7713]), tensor([1.8625]))
train_loader -  (tensor([0.7713]), tensor([1.8625]))
